# Term Structure Binomial Model Solutions


1. **Short-rate binomial lattice**
   - The node $N_{i,j}$ represents time $i$, state $j$.
   - The short-rate $r_{i,j}$ is the risk-free one-period rate from $t=i$ to $t=i+1$.
   - In this question, the recombining lattice is

$$
r_{i,j}=r_{0,0}u^j d^{\,i-j}.
$$

2. **Risk-neutral backward induction**
   - For a non-coupon paying security with value $V_{i,j}$:

$$
V_{i,j}
=
\frac{
q_d V_{i+1,j}+q_u V_{i+1,j+1}
}{
1+r_{i,j}
}.
$$

3. **Zero-coupon bond**
   - At maturity $T$, the terminal value is the face value.
   - Work backward to time 0 using the short rate at each node.

4. **Forward price**
   - For a forward expiring at $T_f$ on an underlying security $S_{T_f}$, the fair delivery price $G_0$ satisfies:

$$
0
=
\mathbb{E}^{Q}_{0}
\left[
\frac{S_{T_f}-G_0}{B_{T_f}}
\right],
$$

so

$$
G_0
=
\frac{
\mathbb{E}^{Q}_{0}\left[S_{T_f}/B_{T_f}\right]
}{
\mathbb{E}^{Q}_{0}\left[1/B_{T_f}\right]
}.
$$

5. **Futures price**
   - For a futures contract expiring at $T_f$:

$$
F_0
=
\mathbb{E}^{Q}_{0}\left[S_{T_f}\right].
$$

6. **American option**
   - At each node before expiry, compare immediate exercise and continuation value.

## Notation

All symbols used in this notebook are defined here once.

### Lattice and rates

| Symbol | Meaning |
|---|---|
| $i$ | Time index (period), $i=0,1,\ldots,n$ |
| $j$ | State index at time $i$, equal to the number of up-moves so far, $j=0,1,\ldots,i$ |
| $N_{i,j}$ | Node of the recombining lattice at time $i$, state $j$ |
| $n$ | Number of periods in the short-rate lattice ($n=10$ here) |
| $r_{i,j}$ | Risk-free one-period short rate at node $(i,j)$, applying from $t=i$ to $t=i+1$ |
| $r_{0,0}$ | Initial short rate at the root node |
| $u$ | Up-move multiplier of the short rate, $r_{i+1,j+1}=u\,r_{i,j}$ |
| $d$ | Down-move multiplier of the short rate, $r_{i+1,j}=d\,r_{i,j}$ |
| $j_t$ | State actually visited at time $t$ along a path through the lattice |

### Probabilities and discounting

| Symbol | Meaning |
|---|---|
| $Q$ | Risk-neutral (martingale) probability measure |
| $q_u$ | Risk-neutral probability of an **up**-move over one period, $q_u=\Pr^Q(j\to j+1)$ |
| $q_d$ | Risk-neutral probability of a **down**-move over one period, $q_d=1-q_u$ |
| $\mathbb{E}^Q_0[\cdot]$ | Expectation under $Q$ conditional on information at time $0$ |
| $B_t$ | Cash account (numeraire) value at time $t$, $B_0=1$ and $B_t=\prod_{s=0}^{t-1}\left(1+r_{s,j_s}\right)$ |

In this problem $q_u$ and $q_d$ are **given** as model inputs:

$$
q_u=q_d=\tfrac12,\qquad q_u+q_d=1 .
$$

They are not derived from $u$ and $d$: in a short-rate lattice the short rate is not a traded
asset, so the lattice is specified directly under $Q$. In code they are `params.q_up` and
`params.q_down`.

### Securities and contracts

| Symbol | Meaning |
|---|---|
| $V_{i,j}$ | Value at node $(i,j)$ of a generic non-coupon-paying security |
| $Z^{(T)}_{i,j}$ | Value at node $(i,j)$ of a zero-coupon bond (ZCB) maturing at $T$ with face value $100$ |
| $P(0,T;\text{face})$ | Time-0 price of a ZCB maturing at $T$ with the stated face value |
| $T_m$ | Maturity of the underlying ZCB ($T_m=10$) |
| $S_{i,j}$ | Value of the underlying security at node $(i,j)$; here $S_{i,j}=Z^{(10)}_{i,j}$ |
| $T_f$ | Expiration of the forward / futures contract ($T_f=4$) |
| $G_0$ | Fair forward (delivery) price agreed at time 0 for delivery at $T_f$ |
| $F_0$ | Initial futures price for expiration $T_f$ |
| $T_o$ | Expiration of the option ($T_o=6$) |
| $K$ | Option strike price ($K=80$) |
| $C_{i,j}$ | American call value at node $(i,j)$ |
| $E_{i,j}$ | Immediate exercise value, $E_{i,j}=\max(S_{i,j}-K,0)$ |
| $H_{i,j}$ | Continuation (hold) value, the discounted risk-neutral expectation of $C_{i+1,\cdot}$ |



## Problem setup

Given:

$$
n=10,\quad r_{0,0}=5\%,\quad u=1.1,\quad d=0.9,\quad q_u=q_d=\frac12
$$

The underlying ZCB has:

$$
T_m=10,\quad \text{face value}=100
$$

For Questions 2 and 3:

$$
T_f=4
$$

For Question 4:

$$
T_o=6,\quad K=80
$$

Important interpretation:

- Question 2 asks for the **fair forward price** $G_0$, because no delivery price is given. If the forward is entered at this fair delivery price, its initial contract value is zero.
- Question 3 asks for the **initial futures price** $F_0$. The initial value of entering the futures contract is zero, but the quoted futures price is $F_0$.


In [1]:

import math
from dataclasses import dataclass

import numpy as np
import pandas as pd


In [2]:

@dataclass(frozen=True)
class BinomialShortRateParams:
    n: int = 10
    r0: float = 0.05
    u: float = 1.1
    d: float = 0.9
    q_up: float = 0.5

    @property
    def q_down(self) -> float:
        return 1.0 - self.q_up


def build_short_rate_lattice(params: BinomialShortRateParams) -> list[list[float]]:
    # Build r_{i,j} = r0 * u^j * d^(i-j).
    # i = time index; j = number of up moves at time i.
    return [
        [
            params.r0 * (params.u ** j) * (params.d ** (i - j))
            for j in range(i + 1)
        ]
        for i in range(params.n + 1)
    ]


def price_zcb_lattice(
    rates: list[list[float]],
    maturity: int,
    face: float = 100.0,
    q_up: float = 0.5,
) -> list[list[float]]:
    # Price a zero-coupon bond by backward induction.
    # Terminal condition: Z_{T,j} = face.
    # Recursion:
    # Z_{i,j} = (q_down * Z_{i+1,j} + q_up * Z_{i+1,j+1}) / (1 + r_{i,j})
    q_down = 1.0 - q_up
    values: list[list[float] | None] = [None] * (maturity + 1)
    values[maturity] = [float(face)] * (maturity + 1)

    for i in range(maturity - 1, -1, -1):
        values[i] = [
            (
                q_down * values[i + 1][j]
                + q_up * values[i + 1][j + 1]
            )
            / (1.0 + rates[i][j])
            for j in range(i + 1)
        ]

    return values


def binomial_expectation_at_time(
    state_values: list[float],
    q_up: float = 0.5,
) -> float:
    # Compute E_Q[value at time t] from t=0 using binomial probabilities.
    t = len(state_values) - 1
    q_down = 1.0 - q_up

    return sum(
        math.comb(t, j) * (q_up ** j) * (q_down ** (t - j)) * state_values[j]
        for j in range(t + 1)
    )


def price_american_call_on_zcb(
    rates: list[list[float]],
    underlying_zcb_lattice: list[list[float]],
    expiration: int,
    strike: float,
    q_up: float = 0.5,
) -> tuple[list[list[float]], list[list[float]], list[list[float] | None]]:
    # Price an American call option on a ZCB by backward induction.
    # Returns option values, exercise values, and continuation values.
    q_down = 1.0 - q_up

    option_values: list[list[float] | None] = [None] * (expiration + 1)
    exercise_values: list[list[float] | None] = [None] * (expiration + 1)
    continuation_values: list[list[float] | None] = [None] * (expiration + 1)

    option_values[expiration] = [
        max(underlying_zcb_lattice[expiration][j] - strike, 0.0)
        for j in range(expiration + 1)
    ]
    exercise_values[expiration] = [
        underlying_zcb_lattice[expiration][j] - strike
        for j in range(expiration + 1)
    ]
    continuation_values[expiration] = None

    for i in range(expiration - 1, -1, -1):
        continuation_values[i] = [
            (
                q_down * option_values[i + 1][j]
                + q_up * option_values[i + 1][j + 1]
            )
            / (1.0 + rates[i][j])
            for j in range(i + 1)
        ]

        exercise_values[i] = [
            underlying_zcb_lattice[i][j] - strike
            for j in range(i + 1)
        ]

        option_values[i] = [
            max(exercise_values[i][j], continuation_values[i][j], 0.0)
            for j in range(i + 1)
        ]

    return option_values, exercise_values, continuation_values


def lattice_to_dataframe(
    lattice: list[list[float]],
    decimals: int = 6,
    percent: bool = False,
) -> pd.DataFrame:
    # Convert a triangular lattice into a rectangular DataFrame.
    rows = []
    for i, row in enumerate(lattice):
        for j, value in enumerate(row):
            rows.append(
                {
                    "time_i": i,
                    "state_j": j,
                    "value": round(value * 100, decimals) if percent else round(value, decimals),
                }
            )
    return pd.DataFrame(rows)


## Build the short-rate lattice

In [3]:

params = BinomialShortRateParams(
    n=10,
    r0=0.05,
    u=1.1,
    d=0.9,
    q_up=0.5,
)

rates = build_short_rate_lattice(params)

rate_df = lattice_to_dataframe(rates, decimals=4, percent=True)
rate_df.head(15)


,time_i,state_j,value
0,0,0,5.0000
1,1,0,4.5000
2,1,1,5.5000
3,2,0,4.0500
4,2,1,4.9500
5,2,2,6.0500
6,3,0,3.6450
7,3,1,4.4550
8,3,2,5.4450
9,3,3,6.6550


## Question 1 - Price of the 10-period ZCB

We price the zero-coupon bond as $Z^{(10)}_{i,j}$, where the superscript means the bond matures at $T=10$.

### Step 1: Build the short-rate lattice

The rate at node $(i,j)$ is

$$
r_{i,j}
=
0.05(1.1)^j(0.9)^{i-j}.
$$

For example,

$$
r_{9,0}=0.05(0.9)^9=0.0193710245,
$$

and

$$
r_{9,9}=0.05(1.1)^9=0.1178973846.
$$

### Step 2: Set the terminal ZCB payoff

Because this is a zero-coupon bond with face value 100,

$$
Z^{(10)}_{10,j}=100,
\quad j=0,1,\ldots,10.
$$

### Step 3: Work backward one node at a time

The risk-neutral probabilities are

$$
q_u=q_d=0.5.
$$

Therefore,

$$
Z^{(10)}_{i,j}
=
\frac{
0.5Z^{(10)}_{i+1,j}
+
0.5Z^{(10)}_{i+1,j+1}
}{
1+r_{i,j}
}.
$$

A one-step example at node $(9,0)$ is

$$
Z^{(10)}_{9,0}
=
\frac{0.5(100)+0.5(100)}{1+0.05(0.9)^9}
=
98.099708.
$$

Then one more step backward at node $(8,0)$ is

$$
Z^{(10)}_{8,0}
=
\frac{
0.5Z^{(10)}_{9,0}
+
0.5Z^{(10)}_{9,1}
}{
1+r_{8,0}
}
=
\frac{
0.5(98.099708)
+
0.5(97.687188)
}{
1+0.0215233605
}
=
95.830846.
$$

Repeating this backward induction from $i=9$ down to $i=0$ gives

$$
Z^{(10)}_{0,0}
=
61.6219581175.
$$

### Equivalent expectation formula

The same answer can be written as the discounted risk-neutral expectation:

$$
Z^{(10)}_{0,0}
=
100\,
\mathbb{E}^Q_0
\left[
\frac{1}{B_{10}}
\right],
$$

where

$$
B_{10}
=
\prod_{t=0}^9
\left(1+r_{t,j_t}\right).
$$

So the numerical result is

$$
\boxed{Z^{(10)}_{0,0}=61.6219581175\approx 61.62}.
$$

In [4]:

zcb10 = price_zcb_lattice(
    rates=rates,
    maturity=10,
    face=100.0,
    q_up=params.q_up,
)

q1_zcb_price = zcb10[0][0]

print(f"Question 1 raw price: {q1_zcb_price:.10f}")
print(f"Question 1 submission answer: {q1_zcb_price:.2f}")

# Show selected ZCB prices from the lattice
zcb10_df = lattice_to_dataframe(zcb10, decimals=6)
zcb10_df.head(20)


Question 1 raw price: 61.6219581175
Question 1 submission answer: 61.62


,time_i,state_j,value
0,0,0,61.621958
1,1,0,67.441030
2,1,1,61.965082
3,2,0,72.881830
4,2,1,68.069922
5,2,2,62.676402
6,3,0,77.886862
7,3,1,73.780227
8,3,2,69.098538
9,3,3,63.838111


## Question 2 - Forward price on the same ZCB, forward expiration $T_f=4$

The underlying security is the same ZCB that matures at $T_m=10$ and has face value 100.

Let

$$
S_4 = Z^{(10)}_{4,j},
$$

the value at time 4 of the ZCB that pays 100 at time 10.

### Step 1: Use the fair forward pricing equation

At initiation, the fair forward has value 0:

$$
0
=
\mathbb{E}^Q_0
\left[
\frac{S_4-G_0}{B_4}
\right].
$$

Rearrange:

$$
\mathbb{E}^Q_0
\left[
\frac{S_4}{B_4}
\right]
-
G_0
\mathbb{E}^Q_0
\left[
\frac{1}{B_4}
\right]
=0.
$$

Therefore,

$$
G_0
=
\frac{
\mathbb{E}^Q_0\left[S_4/B_4\right]
}{
\mathbb{E}^Q_0\left[1/B_4\right]
}.
$$

### Step 2: Use the ZCB identity

Because $S_4$ is the time-4 value of a ZCB that pays 100 at time 10,

$$
\mathbb{E}^Q_0\left[\frac{S_4}{B_4}\right]
=
P(0,10;\text{face}=100).
$$

Also,

$$
\mathbb{E}^Q_0\left[\frac{1}{B_4}\right]
=
P(0,4;\text{face}=1).
$$

So

$$
G_0
=
\frac{
P(0,10;\text{face}=100)
}{
P(0,4;\text{face}=1)
}.
$$

### Step 3: Substitute the computed values

From Question 1,

$$
P(0,10;\text{face}=100)
=
61.6219581175.
$$

Using the same ZCB recursion with maturity $4$ and face value $1$:

$$
P(0,4;\text{face}=1)
=
0.8228895736.
$$

Therefore,

$$
G_0
=
\frac{61.6219581175}{0.8228895736}
=
74.8848449384.
$$

So the answer is

$$
\boxed{G_0=74.8848449384\approx 74.88}.
$$

This is the **fair delivery price** of the forward. The initial value of entering this forward contract at this delivery price is 0.

In [5]:

zcb4_unit = price_zcb_lattice(
    rates=rates,
    maturity=4,
    face=1.0,
    q_up=params.q_up,
)

p_0_10_face100 = zcb10[0][0]
p_0_4_face1 = zcb4_unit[0][0]

q2_forward_price = p_0_10_face100 / p_0_4_face1

print(f"P(0,10; face=100): {p_0_10_face100:.10f}")
print(f"P(0,4; face=1):   {p_0_4_face1:.10f}")
print(f"Question 2 raw forward price: {q2_forward_price:.10f}")
print(f"Question 2 submission answer: {q2_forward_price:.2f}")


P(0,10; face=100): 61.6219581175
P(0,4; face=1):   0.8228895736
Question 2 raw forward price: 74.8848449384
Question 2 submission answer: 74.88


## Question 3 - Initial futures price on the same ZCB, futures expiration $T_f=4$

The futures contract expires at time 4. At expiration, the futures price must equal the underlying ZCB value:

$$
F_4 = S_4.
$$

For futures, the price is obtained by taking the risk-neutral expectation without discounting:

$$
F_0
=
\mathbb{E}^Q_0[S_4].
$$

Since $q_u=q_d=0.5$, the probability of reaching state $j$ at time 4 is

$$
\Pr(j\text{ up moves in 4 periods})
=
\binom{4}{j}(0.5)^j(0.5)^{4-j}
=
\binom{4}{j}(0.5)^4.
$$

The time-4 ZCB values $S_4=Z^{(10)}_{4,j}$ are taken from the ZCB lattice computed in Question 1.

| State $j$ | $S_{4,j}=Z^{(10)}_{4,j}$ | Probability $\binom{4}{j}(0.5)^4$ | Contribution |
|---:|---:|---:|---:|
| 0 | 82.422216 | 0.0625 | 5.151389 |
| 1 | 79.029459 | 0.2500 | 19.757365 |
| 2 | 75.104814 | 0.3750 | 28.164305 |
| 3 | 70.617093 | 0.2500 | 17.654273 |
| 4 | 65.555982 | 0.0625 | 4.097249 |

Therefore,

$$
\begin{aligned}
F_0
&=
\sum_{j=0}^4
\binom{4}{j}(0.5)^4 Z^{(10)}_{4,j} \\
&=
74.8245806314.
\end{aligned}
$$

So the answer is

$$
\boxed{F_0=74.8245806314\approx 74.82}.
$$

Notice that this is close to the forward price from Question 2, but it is not exactly the same because futures are marked to market through time.

In [6]:

s4_values = zcb10[4]
q3_futures_price = binomial_expectation_at_time(
    state_values=s4_values,
    q_up=params.q_up,
)

s4_df = pd.DataFrame({
    "state_j": list(range(5)),
    "S_4 = ZCB value at t=4": s4_values,
    "binomial_probability": [
        math.comb(4, j) * (params.q_up ** j) * (params.q_down ** (4 - j))
        for j in range(5)
    ],
})
s4_df["probability_x_value"] = (
    s4_df["S_4 = ZCB value at t=4"] * s4_df["binomial_probability"]
)

print(f"Question 3 raw futures price: {q3_futures_price:.10f}")
print(f"Question 3 submission answer: {q3_futures_price:.2f}")

s4_df


Question 3 raw futures price: 74.8245806314
Question 3 submission answer: 74.82


,state_j,S_4 = ZCB value at t=4,binomial_probability,probability_x_value
0,0,82.422216,0.0625,5.151389
1,1,79.029459,0.2500,19.757365
2,2,75.104814,0.3750,28.164305
3,3,70.617093,0.2500,17.654273
4,4,65.555982,0.0625,4.097249


## Question 4 - American call option on the same ZCB

The option is an American call on the same ZCB. The underlying is

$$
S_{i,j}=Z^{(10)}_{i,j}.
$$

The option has

$$
T_o=6,
\quad
K=80.
$$

### Step 1: Terminal payoff at expiration

At time 6, the option value is the exercise payoff:

$$
C_{6,j}
=
\max\left(Z^{(10)}_{6,j}-80,0\right).
$$

Using the ZCB lattice:

| State $j$ | $Z^{(10)}_{6,j}$ | Exercise payoff $\max(Z^{(10)}_{6,j}-80,0)$ |
|---:|---:|---:|
| 0 | 90.047460 | 10.047460 |
| 1 | 88.007901 | 8.007901 |
| 2 | 85.593596 | 5.593596 |
| 3 | 82.755094 | 2.755094 |
| 4 | 79.445079 | 0.000000 |
| 5 | 75.622898 | 0.000000 |
| 6 | 71.260629 | 0.000000 |

### Step 2: Backward induction with early exercise

For each node before expiration, calculate:

Immediate exercise value:

$$
E_{i,j}
=
\max\left(Z^{(10)}_{i,j}-80,0\right).
$$

Continuation value:

$$
H_{i,j}
=
\frac{
0.5C_{i+1,j}
+
0.5C_{i+1,j+1}
}{
1+r_{i,j}
}.
$$

American option value:

$$
C_{i,j}
=
\max\left(E_{i,j},H_{i,j}\right).
$$

### Step 3: Example node calculations

At node $(5,0)$:

$$
E_{5,0}
=
\max(86.474562-80,0)
=
6.474562.
$$

The continuation value is

$$
H_{5,0}
=
\frac{
0.5(10.047460)
+
0.5(8.007901)
}{
1+0.0295245000
}
=
8.768786.
$$

So

$$
C_{5,0}
=
\max(6.474562,8.768786)
=
8.768786.
$$

At the root node $(0,0)$, immediate exercise is

$$
E_{0,0}
=
\max(61.621958-80,0)
=
0.
$$

The continuation value is

$$
H_{0,0}
=
\frac{
0.5C_{1,0}
+
0.5C_{1,1}
}{
1+r_{0,0}
}
=
\frac{
0.5(3.393500)
+
0.5(1.556652)
}{
1+0.05
}
=
2.3572151638.
$$

Therefore,

$$
C_{0,0}
=
\max(0,2.3572151638)
=
2.3572151638.
$$

So the answer is

$$
\boxed{C_{0,0}=2.3572151638\approx 2.36}.
$$

The code output below also shows the full exercise-versus-continuation diagnostic table.

In [7]:

option_values, exercise_values, continuation_values = price_american_call_on_zcb(
    rates=rates,
    underlying_zcb_lattice=zcb10,
    expiration=6,
    strike=80.0,
    q_up=params.q_up,
)

q4_american_call_price = option_values[0][0]

print(f"Question 4 raw American call price: {q4_american_call_price:.10f}")
print(f"Question 4 submission answer: {q4_american_call_price:.2f}")

# Create a diagnostic table showing exercise vs continuation.
rows = []
for i in range(7):
    for j in range(i + 1):
        underlying_value = zcb10[i][j]
        exercise_payoff = max(underlying_value - 80.0, 0.0)
        continuation = None if i == 6 else continuation_values[i][j]
        option_value = option_values[i][j]
        early_exercise = (
            i < 6
            and exercise_payoff > 0
            and abs(option_value - exercise_payoff) < 1e-10
        )

        rows.append({
            "time_i": i,
            "state_j": j,
            "underlying_ZCB": underlying_value,
            "exercise_payoff": exercise_payoff,
            "continuation_value": continuation,
            "option_value": option_value,
            "early_exercise_before_expiry": early_exercise,
        })

american_call_df = pd.DataFrame(rows)
american_call_df.round(6)


Question 4 raw American call price: 2.3572151638
Question 4 submission answer: 2.36


,time_i,state_j,underlying_ZCB,exercise_payoff,continuation_value,option_value,early_exercise_before_expiry
0,0,0,61.621958,0.000000,2.357215,2.357215,False
1,1,0,67.441030,0.000000,3.393500,3.393500,False
2,1,1,61.965082,0.000000,1.556652,1.556652,False
3,2,0,72.881830,0.000000,4.647335,4.647335,False
4,2,1,68.069922,0.000000,2.445080,2.445080,False
5,2,2,62.676402,0.000000,0.839455,0.839455,False
6,3,0,77.886862,0.000000,6.030298,6.030298,False
7,3,1,73.780227,0.000000,3.640807,3.640807,False
8,3,2,69.098538,0.000000,1.491416,1.491416,False
9,3,3,63.838111,0.000000,0.289068,0.289068,False


## Final answers

The answers to submit are rounded to two decimals.

| Question | Raw value | Submission answer |
|---|---:|---:|
| Q1: ZCB price, maturity $t=10$, face 100 | $61.6219581175$ | **61.62** |
| Q2: Fair forward price, expiration $t=4$ | $74.8848449384$ | **74.88** |
| Q3: Initial futures price, expiration $t=4$ | $74.8245806314$ | **74.82** |
| Q4: American call price, expiration $t=6$, strike 80 | $2.3572151638$ | **2.36** |

In [8]:

answers = pd.DataFrame({
    "question": [
        "Q1: ZCB price, maturity t=10, face=100",
        "Q2: Fair forward price, expiration t=4",
        "Q3: Initial futures price, expiration t=4",
        "Q4: American call price, expiration t=6, strike=80",
    ],
    "raw_value": [
        q1_zcb_price,
        q2_forward_price,
        q3_futures_price,
        q4_american_call_price,
    ],
})

answers["submission_answer"] = answers["raw_value"].round(2)
answers


,question,raw_value,submission_answer
0,"Q1: ZCB price, maturity t=10, face=100",61.621958,61.62
1,"Q2: Fair forward price, expiration t=4",74.884845,74.88
2,"Q3: Initial futures price, expiration t=4",74.824581,74.82
3,"Q4: American call price, expiration t=6, strik...",2.357215,2.36
